# A2A Customer Support — KG-Guided Activation Steering
## Complete Notebook (vector build + verification + full pipeline + batch runner)

### Requirements

Place these files in the **same directory** as this notebook:
```
main.py
a2a_kg_pipeline_complete (1) spacy.ipynb
```

Set your secrets as environment variables **before** launching:
```bash
export HF_TOKEN="hf_..."         # Hugging Face token (Llama-2 is gated — accept licence first)
export GROQ_API_KEY="gsk_..."    # Groq API key — free at https://console.groq.com
```

Or set them directly in **Cell 2** below.

### What this notebook does
1. Installs all dependencies (including spaCy + `en_core_web_sm`)
2. Authenticates with Hugging Face
3. Loads `meta-llama/Llama-2-13b-chat-hf` **once** (into `_model_cache`) — avoids triple-loading
4. **Patches** `_run_kg_activation_steering` and `_get_activation` for `device_map='auto'` compatibility
5. Builds both style vectors (`empathetic`, `formal`) using PCA on contrast pairs
6. Verifies vectors (norm, shape, cosine similarity assertions)
7. Smoke-tests activation steering (same prompt → two different tones)
8. Lists saved `.pkl` files in `.style_cache/`
9. [Optional] Runs the full A2A pipeline (single scenario)
10. [Optional] Runs the full 100-case batch and saves results to `outputs/`

### Runtime recommendation
**Google Colab A100 GPU** (≥40 GB VRAM). The 13b model requires ~26 GB in bfloat16.  
Runtime → Change runtime type → A100 GPU

### Model change (vs previous version)
| Setting | Old | New |
|---|---|---|
| `LOCAL_MODEL_NAME` | `Llama-2-7b-hf` | `Llama-2-13b-chat-hf` |
| `STEER_LAYER` | 20 | 28 |
| `STEER_ALPHA` | 15.0 | 12.0 |
| Hidden dim | 4096 | 5120 |
| Prompt format | Base `[INST]` | Chat `<s>[INST] <<SYS>>` |

### Bug fixed — `Tensor on device cuda:0 is not on the expected device meta!`
When loading with `device_map='auto'`, accelerate manages layer dispatch across devices and
keeps some weights on `meta` device. The original steering hook pre-captured the device from
`next(model.parameters()).device` which returned `meta`, causing the error. **Cell 6b** patches
both `_run_kg_activation_steering` and `_get_activation` so they resolve the device dynamically
from `hidden.device` inside the hook, eliminating the mismatch.


In [1]:
# ─── Cell 0a — AGGRESSIVE GPU CLEANUP (RUN THIS FIRST EVERY SESSION) ────────
#
# FIXES:
#   1. 'Allocated: 16.93 GB after clearing'  → previous model wasn't fully freed
#   2. '13b offloaded to CPU / meta device'   → not enough VRAM left for full model
#   3. 'Model keeps re-downloading'           → HF cache not persistent across sessions
#
# ALWAYS run this cell before Cell 1 so VRAM is clean before the 13b loads.
# If VRAM still shows > 1 GB after this, do: Runtime → Restart runtime → re-run.

import gc, sys

# ── Step 1: Wipe main_module model cache (the real source of the VRAM leak) ──
if 'main_module' in dir():
    try:
        main_module._model_cache.clear()
        print('  ✓ main_module._model_cache.clear() — done')
    except Exception as e:
        print(f'  main_module cache clear skipped: {e}')

# ── Step 2: Delete model / tokenizer from globals ──────────────────────────
_gone = []
for _v in [k for k in list(globals().keys())
            if any(x in k.lower() for x in ['model', 'tokenizer', 'main_module'])]:
    try:
        del globals()[_v]
        _gone.append(_v)
    except Exception:
        pass
if _gone:
    print(f'  ✓ Deleted globals: {_gone}')

# ── Step 3: Evict main_module from sys.modules so Cell 5 reimports cleanly ──
_unloaded = [k for k in list(sys.modules.keys()) if 'main_module' in k]
for _m in _unloaded:
    del sys.modules[_m]
if _unloaded:
    print(f'  ✓ Unloaded from sys.modules: {_unloaded}')

# ── Step 4: Run garbage collection + flush CUDA cache ──────────────────────
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()          # wait for all CUDA ops to finish
        torch.cuda.reset_peak_memory_stats()

        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        free      = total - reserved   # reserved includes fragmented blocks

        print()
        print(f'  GPU         : {torch.cuda.get_device_name(0)}')
        print(f'  Total VRAM  : {total:.1f} GB')
        print(f'  Allocated   : {allocated:.2f} GB  ← should be ~0.0 GB')
        print(f'  Reserved    : {reserved:.2f} GB')
        print(f'  Approx Free : {free:.1f} GB')
        print()

        if allocated > 1.0:
            print('  ⚠  WARNING: VRAM still in use — a process is holding the GPU.')
            print('  → Runtime → Restart runtime, then re-run from Cell 0a.')
        elif total < 26.0:
            print(f'  ⚠  WARNING: Only {total:.1f} GB total VRAM.')
            print('  Llama-2-13b-chat-hf needs ~26 GB in bfloat16.')
            print('  → Runtime → Change runtime type → A100 GPU (40 GB)')
        else:
            print('  ✓ VRAM is clean and sufficient — safe to load Llama-2-13b.')
    else:
        print('  ⚠  No GPU detected! → Runtime → Change runtime type → A100 GPU')
except ImportError:
    print('  torch not installed yet — will be installed in Cell 1')



  GPU         : NVIDIA RTX 6000 Ada Generation
  Total VRAM  : 50.9 GB
  Allocated   : 0.00 GB  ← should be ~0.0 GB
  Reserved    : 0.00 GB
  Approx Free : 50.9 GB

  ✓ VRAM is clean and sufficient — safe to load Llama-2-13b.


In [2]:
# ─── Cell 0b — MOUNT GOOGLE DRIVE FOR PERSISTENT MODEL CACHE ────────────────
#
# FIXES: 'model keeps re-downloading every session'
#
# Colab's /root/.cache is WIPED on every disconnect. Mounting Drive and
# pointing HF_HOME there means the 26 GB checkpoint shards are downloaded
# ONCE and reused across all future sessions.
#
# First run: downloads ~26 GB to your Drive (takes ~10 min on A100)
# All later runs: loads from Drive cache (~2-3 min instead of ~10 min)
#
# Skip this cell ONLY if you want to use the ephemeral /root/.cache
# (model will re-download every session).

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_CACHE = '/content/drive/MyDrive/hf_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)

    # Set ALL HuggingFace cache env vars before any transformers import
    os.environ['HF_HOME']              = DRIVE_CACHE
    os.environ['TRANSFORMERS_CACHE']   = DRIVE_CACHE
    os.environ['HF_DATASETS_CACHE']    = DRIVE_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE']= DRIVE_CACHE

    # Check if shards already exist in Drive cache
    import glob
    existing = glob.glob(f'{DRIVE_CACHE}/**/*.safetensors', recursive=True)
    existing += glob.glob(f'{DRIVE_CACHE}/**/*.bin', recursive=True)

    print(f'  ✓ HuggingFace cache → {DRIVE_CACHE}')
    if existing:
        total_gb = sum(os.path.getsize(f) for f in existing) / 1e9
        print(f'  ✓ Found {len(existing)} cached weight file(s) ({total_gb:.1f} GB) — NO re-download needed!')
    else:
        print('  ℹ  No cached weights yet — first run will download ~26 GB to Drive.')
        print('  All future sessions will reuse this cache.')

except Exception as e:
    # Not running in Colab, or Drive already mounted
    print(f'  Drive mount skipped ({e}) — using default HF cache at /root/.cache')
    print('  ⚠  Model will re-download every session without Drive cache.')


  Drive mount skipped (No module named 'google.colab') — using default HF cache at /root/.cache
  ⚠  Model will re-download every session without Drive cache.


In [3]:
# ─── Cell 1 — Install all dependencies ───────────────────────────────────────
#
# Installs both the core pipeline deps (fastapi, groq, langchain) AND
# the activation-steering deps (torch, transformers, accelerate, sklearn).
# Also installs spaCy and downloads en_core_web_sm for KG-NER.

%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    pydantic==2.8.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    transformers==4.40.0 \
    accelerate>=0.29.0 \
    huggingface_hub>=0.22.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    spacy

# torch is pre-installed on Colab GPU runtimes; install only if missing
try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch
    print(f"✓ torch installed: {torch.__version__}")

# Download spaCy NER model used by main.py for entity extraction
!python -m spacy download en_core_web_sm

print("✓ All dependencies installed")

Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.6 MB/s  0:00:00m0:00:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ All dependencies installed


In [4]:
# ─── Cell 1b — Verify spaCy NER model ────────────────────────────────────────
#
# Confirms that en_core_web_sm downloaded correctly.
# main.py loads it lazily; this cell lets you catch missing-model errors early.

import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("Priya Sharma ordered TechPro X200 Laptop under ORD-7741.")
print("✓ spaCy en_core_web_sm loaded")
print("  Entities found in test sentence:")
for ent in doc.ents:
    print(f"    {ent.text!r:30s}  label={ent.label_}")
print()
print("  (Expected: PERSON for customer name, PRODUCT/ORG for product name)")

✓ spaCy en_core_web_sm loaded
  Entities found in test sentence:
    'Priya Sharma'                  label=PERSON
    'TechPro'                       label=ORG
    'X200'                          label=PRODUCT

  (Expected: PERSON for customer name, PRODUCT/ORG for product name)


In [ ]:
# ─── Cell 2 — Set secrets and authenticate with Hugging Face ─────────────────
#
# Option A (recommended): export HF_TOKEN and GROQ_API_KEY before launching.
# Option B: fill in the strings below (do NOT commit credentials to git).

import os
from huggingface_hub import login

# ── Uncomment and fill in if not already exported ────────────────────────────
os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found.\n"
        "Run:  export HF_TOKEN='hf_...'\n"
        "Or uncomment the assignment line above."
    )

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY:
    print("✓ GROQ_API_KEY found — full pipeline (Cell 12) will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build/smoke-test will still work; Cell 12 requires it")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline (Cell 12) will work


In [6]:
# ─── Cell 3 — Verify GPU and version compatibility ────────────────────────────

import torch
import transformers

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    if vram_gb < 26.0:
        print("⚠  WARNING: Llama-2-13b-chat-hf needs ~26 GB VRAM in bfloat16.")
        print("   Upgrade to A100: Runtime → Change runtime type → A100 GPU")
    else:
        print("✓ Sufficient VRAM for Llama-2-13b-chat-hf in bfloat16")
else:
    print("⚠  WARNING: No GPU detected — inference will be extremely slow.")
    print("   Runtime → Change runtime type → A100 GPU")

torch        : 2.10.0+cu128
transformers : 4.40.0
CUDA         : True
GPU          : NVIDIA RTX 6000 Ada Generation
VRAM         : 50.9 GB
✓ Sufficient VRAM for Llama-2-13b-chat-hf in bfloat16


In [7]:
# ─── Cell 4 — Verify main.py exists in current directory ─────────────────────

from pathlib import Path

main_file = Path("main.py")
if not main_file.exists():
    raise FileNotFoundError(
        "main.py not found in the current directory.\n"
        f"Current directory: {Path.cwd()}\n"
        "Upload or copy main.py here before proceeding."
    )

size_kb = main_file.stat().st_size / 1024
print(f"✓ Found main.py at : {main_file.resolve()}")
print(f"  Size             : {size_kb:.1f} KB")

✓ Found main.py at : /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-2-13b-chat-hf/main.py
  Size             : 97.1 KB


In [8]:
# ─── Cell 5 — Configure environment variables and import main.py ──────────────
#
# These env vars must be set BEFORE importing main.py because module-level
# code in main.py reads them at import time (e.g. STEER_LAYER, STYLE_CACHE_DIR).
#
# CHANGED vs previous version:
#   LOCAL_MODEL_NAME : Llama-2-7b-hf  → Llama-2-13b-chat-hf  (13B chat model)
#   STEER_LAYER      : 20             → 28                    (≈70% depth in 40-layer model)
#   STEER_ALPHA      : 15.0           → 12.0                  (13b is less sensitive than 7b)

import os
import importlib.util
from pathlib import Path

os.environ.setdefault("LOCAL_MODEL_NAME", "meta-llama/Llama-2-13b-chat-hf")
os.environ.setdefault("STEER_LAYER",      "28")
os.environ.setdefault("STEER_ALPHA",      "12.0")
os.environ.setdefault("STYLE_CACHE_DIR",  ".style_cache")

Path(".style_cache").mkdir(exist_ok=True)

# Dynamically import main.py so all its functions are available as main_module.*
spec        = importlib.util.spec_from_file_location("main_module", "main.py")
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print("✓ main.py imported successfully")
print("=" * 55)
print(f"  Model          : {os.environ['LOCAL_MODEL_NAME']}")
print(f"  Steer layer    : {os.environ['STEER_LAYER']}")
print(f"  Steer alpha    : {os.environ['STEER_ALPHA']}")
print(f"  Cache dir      : {os.environ['STYLE_CACHE_DIR']}")
print(f"  Contrast pairs : empathetic={len(main_module.CONTRAST_PAIRS['empathetic'])}, "
      f"formal={len(main_module.CONTRAST_PAIRS['formal'])}")
print("=" * 55)

# Verify spaCy NER is active (main.py loads it lazily; trigger it now)
nlp_check = main_module._get_nlp()
if nlp_check:
    print("✓ spaCy en_core_web_sm active for KG-NER")
else:
    print("⚠  spaCy unavailable — regex-only NER fallback active")

✓ main.py imported successfully
  Model          : meta-llama/Llama-2-13b-chat-hf
  Steer layer    : 28
  Steer alpha    : 12.0
  Cache dir      : .style_cache
  Contrast pairs : empathetic=16, formal=16
✓ spaCy en_core_web_sm active for KG-NER


In [10]:
# ─── Cell 6 — Load Llama-2-13b-chat-hf ONCE (GPU-first, no CPU offload) ──────
#
# FIXED vs original:
#   device_map='auto'  →  device_map={'': 0}  (forces ALL layers onto GPU 0)
#   This eliminates 'Some parameters on meta device (offloaded to CPU)'
#   and the associated VRAM leak when you try to clear between models.
#
#   max_memory guard: if total VRAM < 28 GB the cell falls back to 'auto'
#   with max_memory set so GPU gets priority and CPU gets the remainder.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.environ["LOCAL_MODEL_NAME"]
HF_TOKEN   = os.environ["HF_TOKEN"]

# ── Decide device strategy based on available VRAM ─────────────────────────
if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram_gb  = (torch.cuda.get_device_properties(0).total_memory
                     - torch.cuda.memory_reserved()) / 1e9
    print(f'  GPU total VRAM  : {total_vram_gb:.1f} GB')
    print(f'  VRAM free now   : {free_vram_gb:.1f} GB')
else:
    total_vram_gb = 0
    free_vram_gb  = 0

MODEL_NEEDS_GB = 26.5   # Llama-2-13b in bfloat16

if free_vram_gb >= MODEL_NEEDS_GB:
    # ── BEST PATH: entire model on GPU, zero CPU offload ─────────────────────
    device_map_arg = {'': 0}   # all layers → GPU 0
    max_memory_arg = None
    print('  Strategy : GPU-only (device_map={"":0}) — no CPU offload')
elif total_vram_gb >= MODEL_NEEDS_GB:
    # Free VRAM is tight but total should be enough; re-check after cleanup
    print('  ⚠  Free VRAM is tight but total capacity looks sufficient.')
    print('  → Re-run Cell 0a to free any reserved memory, then retry Cell 6.')
    print('  Falling back to max_memory-constrained auto map (GPU-priority).')
    gpu_limit = f'{int(total_vram_gb - 2)}GiB'   # leave 2 GB headroom for activations
    device_map_arg = 'auto'
    max_memory_arg = {0: gpu_limit, 'cpu': '4GiB'}  # minimise CPU layers
    print(f'  Strategy        : auto + max_memory={{GPU: {gpu_limit}, cpu: 4GiB}}')
else:
    print(f'  ⚠  ERROR: Only {total_vram_gb:.1f} GB total VRAM.')
    print(f'  Llama-2-13b needs ~{MODEL_NEEDS_GB} GB. Upgrade to A100 40 GB.')
    print('  Proceeding with auto + CPU fallback (will be SLOW).')
    device_map_arg = 'auto'
    max_memory_arg = None

print(f'\nLoading {MODEL_NAME} ...')
print('(26 GB download on first run; ~2-3 min from Drive cache on later runs)')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Build from_pretrained kwargs
_load_kwargs = dict(
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map=device_map_arg,
    token=HF_TOKEN,
)
if max_memory_arg is not None:
    _load_kwargs['max_memory'] = max_memory_arg

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_load_kwargs)
model.eval()

# Share with main_module so _get_model_and_tokenizer() never reloads
main_module._model_cache['model']     = model
main_module._model_cache['tokenizer'] = tokenizer

# ── Post-load VRAM report ────────────────────────────────────────────────────
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_res  = torch.cuda.memory_reserved()  / 1e9
    print()
    print(f'✓ {MODEL_NAME} loaded')
    print(f'  VRAM allocated  : {vram_used:.2f} GB')
    print(f'  VRAM reserved   : {vram_res:.2f} GB')

    # Check whether any layers ended up on CPU (meta device)
    cpu_params = sum(1 for p in model.parameters() if p.device.type in ('cpu', 'meta'))
    gpu_params = sum(1 for p in model.parameters() if p.device.type == 'cuda')
    print(f'  GPU params      : {gpu_params}')
    if cpu_params > 0:
        print(f'  ⚠  CPU/meta params : {cpu_params} — some layers offloaded to CPU!')
        print('  If this causes errors, restart runtime, run Cell 0a, and retry.')
    else:
        print('  ✓ All parameters on GPU — no CPU offload!')

n_layers = len(model.model.layers) if hasattr(model, 'model') and hasattr(model.model, 'layers') else '?'
hidden   = model.config.hidden_size if hasattr(model, 'config') else '?'
print(f'  Layers          : {n_layers}  (STEER_LAYER={os.environ["STEER_LAYER"]})')
print(f'  Hidden dim      : {hidden}  (expected 5120 for 13b)')
print(f'  _model_cache    : populated ✓')


  GPU total VRAM  : 50.9 GB
  VRAM free now   : 50.9 GB
  Strategy : GPU-only (device_map={"":0}) — no CPU offload

Loading meta-llama/Llama-2-13b-chat-hf ...
(26 GB download on first run; ~2-3 min from Drive cache on later runs)


/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✓ meta-llama/Llama-2-13b-chat-hf loaded
  VRAM allocated  : 26.24 GB
  VRAM reserved   : 26.25 GB
  GPU params      : 363
  ✓ All parameters on GPU — no CPU offload!
  Layers          : 40  (STEER_LAYER=28)
  Hidden dim      : 5120  (expected 5120 for 13b)
  _model_cache    : populated ✓


In [11]:
# ─── Cell 6b — Patch activation steering for device_map='auto' ──────────────
#
# BUG FIXED: RuntimeError: Tensor on device cuda:0 is not on the expected device meta!
#
# Root cause:
#   device_map='auto' lets accelerate dispatch layers across devices.
#   next(model.parameters()).device can return 'meta' for offloaded parameters.
#   The original hook pre-captured this 'meta' device and moved steer_vec there,
#   but hidden_states were on cuda:0 — causing the device mismatch at multiply.
#
# Fix applied to both functions:
#   _get_activation           — resolves real device by skipping meta params
#   _run_kg_activation_steering — hook moves vec to hidden.device dynamically
#                                 (never pre-captures device outside the hook)

import torch


def _patched_get_activation(model, tokenizer, text: str):
    """
    Captures the last-token hidden state at STEER_LAYER.
    Fixed for device_map='auto': resolves the first real (non-meta) parameter
    device instead of blindly calling next(model.parameters()).device.
    """
    captured = {}

    def _hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type != 'meta':          # skip accelerate meta-device pass-throughs
            captured["act"] = hidden.detach().cpu().float()

    layer  = main_module._get_layer(model)
    handle = layer.register_forward_hook(_hook)
    try:
        # Find the first parameter that lives on a real device (not meta)
        real_device = next(
            (p.device for p in model.parameters() if p.device.type != 'meta'),
            torch.device('cpu')
        )
        inputs = tokenizer(
            text, return_tensors="pt", padding=False, return_attention_mask=True
        ).to(real_device)
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()

    act = captured.get("act")
    if act is None:
        raise RuntimeError(
            "[_get_activation] Activation not captured — "
            "all parameters may be on meta device."
        )
    return act[0, -1, :]


def _patched_run_kg_activation_steering(
    defactualized_prompt: str, style: str, kg
) -> str:
    """
    Steered generation using the activation steering vector.
    Fixed for device_map='auto': vec is moved to hidden.device INSIDE the hook
    so it always matches wherever accelerate has placed the hidden states.
    """
    steer_vec = main_module.get_style_vector(style)
    if steer_vec is None:
        raise RuntimeError(
            f"No style vector for '{style}'. "
            "Run Cells 8–9 to build vectors first."
        )

    model, tokenizer = main_module._get_model_and_tokenizer()

    # Resolve a real device for tokenised inputs
    real_device = next(
        (p.device for p in model.parameters() if p.device.type != 'meta'),
        torch.device('cuda:0')
    )

    style_prefix = {
        "empathetic": (
            "I'm truly sorry to hear about this and I completely understand "
            "how frustrated you must be feeling right now."
        ),
        "formal": (
            "We acknowledge receipt of your complaint and wish to advise you "
            "that your case has been logged and assigned for review."
        ),
    }.get(style, "Thank you for contacting us regarding this matter.")

    system_instruction = (
        f"You are a professional customer support agent. "
        f"Write a {style.upper()} support reply. "
        f"Do NOT use headers or bullet points. "
        f"Write ONLY the reply — 3 to 4 sentences maximum. "
        f"Do NOT analyse the message. Just write the reply."
    )
    full_prompt = (
        f"<s>[INST] <<SYS>>\n{system_instruction}\n<</SYS>>\n\n"
        f"Customer message: {defactualized_prompt} [/INST] "
        f"{style_prefix}"
    )

    inputs     = tokenizer(
        full_prompt, return_tensors="pt", padding=False, return_attention_mask=True
    ).to(real_device)
    input_ids  = inputs.input_ids
    STEER_ALPHA = main_module.STEER_ALPHA
    target_layer = main_module._get_layer(model)

    # KEY FIX: vec is NOT moved to a pre-captured device.
    # Inside the hook we call vec.to(hidden.device) each time, so it
    # always matches whatever device accelerate placed the hidden states on.
    def _steer_hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type == 'meta':          # accelerate dispatch artefact — skip
            return output
        hidden_f32 = hidden.float()
        v = steer_vec.to(device=hidden_f32.device, dtype=torch.float32)  # dynamic device
        hidden_f32 = hidden_f32 + STEER_ALPHA * v
        steered = hidden_f32.to(hidden.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = target_layer.register_forward_hook(_steer_hook)
    try:
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
                eos_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()

    new_ids   = output_ids[0][input_ids.shape[1]:]
    result    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    sentences = [s.strip() for s in result.split(".") if s.strip()]
    if len(sentences) > 4:
        result = ". ".join(sentences[:4]) + "."
    return result


# ── Apply patches to main_module ──────────────────────────────────────────────
main_module._get_activation             = _patched_get_activation
main_module._run_kg_activation_steering = _patched_run_kg_activation_steering

print("✓ main_module._get_activation             — patched (device_map='auto' safe)")
print("✓ main_module._run_kg_activation_steering — patched (vec.to(hidden.device) in hook)")
print()
print("  Both patches survive for the entire session — re-run this cell if you")
print("  reimport main_module (Cell 5) without restarting the kernel.")


✓ main_module._get_activation             — patched (device_map='auto' safe)
✓ main_module._run_kg_activation_steering — patched (vec.to(hidden.device) in hook)

  Both patches survive for the entire session — re-run this cell if you
  reimport main_module (Cell 5) without restarting the kernel.


In [12]:
# ─── Cell 7 — Helper: build a style vector using the already-loaded model ─────
#
# FIX: The original build_style_vector() in main.py always reloads the model
# from disk. This wrapper reuses the model/tokenizer loaded in Cell 6.

import torch
from pathlib import Path

def build_vector_with_loaded_model(style: str, method: str = "pca") -> torch.Tensor:
    """
    Build and cache a unit-norm style direction vector.

    Uses the model + tokenizer already loaded in Cell 6 (no re-download).
    Saves the result to .style_cache/style_vec_{style}_{method}.pkl.

    Parameters
    ----------
    style  : 'empathetic' or 'formal'
    method : 'pca' (recommended) | 'mean' | 'logistic'

    Returns
    -------
    torch.Tensor of shape [hidden_dim] on CPU, unit-norm
    """
    pairs = main_module.CONTRAST_PAIRS.get(style)
    if not pairs:
        raise ValueError(f"No contrast pairs for style '{style}'. "
                         f"Available: {list(main_module.CONTRAST_PAIRS.keys())}")

    pos_acts, neg_acts = [], []
    for i, (pos_text, neg_text) in enumerate(pairs):
        print(f"  Processing pair {i+1:2d}/{len(pairs)} ...", end="\r")
        pos_acts.append(main_module._get_activation(model, tokenizer, pos_text))
        neg_acts.append(main_module._get_activation(model, tokenizer, neg_text))

    print(f"  {len(pairs)}/{len(pairs)} pairs processed              ")

    method_fns = {
        "pca":      main_module._pca,
        "mean":     main_module._mean_difference,
        "logistic": main_module._logistic_regression,
    }
    if method not in method_fns:
        raise ValueError(f"Unknown method '{method}'. Choose from: {list(method_fns)}")

    vec = method_fns[method](pos_acts, neg_acts)
    main_module._save_vector(vec, style, method)
    return vec

print("✓ build_vector_with_loaded_model() defined")

✓ build_vector_with_loaded_model() defined


In [13]:
# ─── Cell 8 — Build EMPATHETIC style vector ───────────────────────────────────

print("Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~2–4 min on A100")
print()

vec_emp = build_vector_with_loaded_model(style="empathetic", method="pca")

print(f"\n✓ EMPATHETIC vector built")
print(f"  shape : {list(vec_emp.shape)}")
print(f"  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_emp.dtype}")

Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...
Expected time: ~2–4 min on A100

  16/16 pairs processed              

✓ EMPATHETIC vector built
  shape : [5120]
  norm  : 0.999999  (should be ~1.0)
  dtype : torch.float32


In [14]:
# ─── Cell 9 — Build FORMAL style vector ──────────────────────────────────────

print("Building FORMAL vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~2–4 min on A100")
print()

# FIX: original notebook had 'vec_formal' here but then referenced 'vec_form' in
# verification and smoke-test cells. Variable is now consistently named 'vec_form'.
vec_form = build_vector_with_loaded_model(style="formal", method="pca")

print(f"\n✓ FORMAL vector built")
print(f"  shape : {list(vec_form.shape)}")
print(f"  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_form.dtype}")

Building FORMAL vector (16 contrast pairs × 2 activations) ...
Expected time: ~2–4 min on A100

  16/16 pairs processed              

✓ FORMAL vector built
  shape : [5120]
  norm  : 0.999999  (should be ~1.0)
  dtype : torch.float32


In [15]:
# ─── Cell 10 — Verify both vectors ───────────────────────────────────────────
#
# CHANGED: assertion updated from hidden_dim=4096 (Llama-2-7b) to
#          hidden_dim=5120 (Llama-2-13b-chat-hf).
#
# FIX retained: 'vec_form' naming is consistent across Cells 9, 10, 11.

import torch

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print("=" * 55)
print(f"  empathetic : norm={vec_emp.norm():.6f}   shape={list(vec_emp.shape)}")
print(f"  formal     : norm={vec_form.norm():.6f}   shape={list(vec_form.shape)}")
print(f"  cosine sim : {sim:.4f}")
print("=" * 55)

# ── Assertions ────────────────────────────────────────────────────────────────
assert abs(vec_emp.norm().item()  - 1.0) < 0.01, \
    f"empathetic vector not unit-norm! norm={vec_emp.norm():.4f}"
assert abs(vec_form.norm().item() - 1.0) < 0.01, \
    f"formal vector not unit-norm! norm={vec_form.norm():.4f}"
assert vec_emp.shape[0]  == 5120, \
    f"Expected hidden_dim=5120 for Llama-2-13b, got {vec_emp.shape[0]}"
assert vec_form.shape[0] == 5120, \
    f"Expected hidden_dim=5120 for Llama-2-13b, got {vec_form.shape[0]}"
assert sim < 0.5, \
    f"Cosine similarity {sim:.3f} too high — vectors may not be distinct directions!"

print()
if sim < 0:
    print("✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (ideal)")
else:
    print(f"  Cosine similarity {sim:.3f} > 0 — vectors diverge but are not anti-parallel.")
    print("  This is acceptable as long as sim < 0.5 (styles are distinct directions).")

import os
cache_files = sorted(os.listdir(".style_cache"))
print(f"\nCached files in .style_cache/:")
for f in cache_files:
    size = os.path.getsize(f".style_cache/{f}") / 1024
    print(f"  {f}  ({size:.1f} KB)")
print("  Note: ~20 KB each = 5120 float32 values (Llama-2-13b hidden dim)")
print()
print("✓ All assertions passed!")

  empathetic : norm=0.999999   shape=[5120]
  formal     : norm=0.999999   shape=[5120]
  cosine sim : -0.9781

✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (ideal)

Cached files in .style_cache/:
  style_vec_empathetic_pca.pkl  (20.4 KB)
  style_vec_formal_pca.pkl  (20.4 KB)
  Note: ~20 KB each = 5120 float32 values (Llama-2-13b hidden dim)

✓ All assertions passed!


In [16]:
# ─── Cell 11 — Smoke test: activation steering changes tone ──────────────────
#
# CHANGED vs previous version:
#   1. TEST_PROMPT uses Llama-2-chat format (<s>[INST] <<SYS>>...)
#   2. ALPHA_TEST = 12.0 (matches STEER_ALPHA for 13b in main.py)
#   3. Hook uses hidden.device dynamically — consistent with Cell 6b patch
#   4. Real device resolved by skipping meta parameters

import torch

ALPHA_TEST = 12.0   # matches STEER_ALPHA in main.py for Llama-2-13b

# Llama-2-chat format
TEST_PROMPT = (
    "<s>[INST] <<SYS>>\n"
    "You are a professional customer support agent. Write a support reply.\n"
    "<</SYS>>\n\n"
    "Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] "
)


def generate_steered(prompt: str, style_vec: torch.Tensor, alpha: float, label: str) -> str:
    """Smoke-test helper. Uses same device-safe pattern as Cell 6b patch."""
    # Resolve first real (non-meta) device — safe under device_map='auto'
    real_device = next(
        (p.device for p in model.parameters() if p.device.type != 'meta'),
        torch.device('cuda:0')
    )
    layer  = main_module._get_layer(model)

    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        if h.device.type == 'meta':
            return output
        h_f32 = h.float()
        v = style_vec.to(device=h_f32.device, dtype=torch.float32)   # dynamic device
        h_f32 = h_f32 + alpha * v
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(real_device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f"\n[{label}]")
    print(f"  {text[:350]}")
    return text


print("=" * 60)
print("Smoke Test: same prompt — two style vectors — different tones")
print("=" * 60)
print(f"Prompt (truncated): {TEST_PROMPT[:120]}...")

r_emp  = generate_steered(TEST_PROMPT, vec_emp,  ALPHA_TEST, "EMPATHETIC")
r_form = generate_steered(TEST_PROMPT, vec_form, ALPHA_TEST, "FORMAL")

print()
print("=" * 60)
print("✓ Smoke test complete")
print("  Confirm that EMPATHETIC response uses warm/personal language")
print("  and FORMAL response uses procedural/institutional language.")
print("=" * 60)


Smoke Test: same prompt — two style vectors — different tones
Prompt (truncated): <s>[INST] <<SYS>>
You are a professional customer support agent. Write a support reply.
<</SYS>>

Customer message: My <...


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



[EMPATHETIC]
  Dear valued customer,

Thank you for reaching out to us regarding your recent order with our company. We apologize for any inconvenience or issues you have experienced with your purchase. Our team is here to help and we will do everything possible to resolve the issue as quickly and efficiently as possible.

Could you please provide more details ab

[FORMAL]
  Dear valued customer,

Thank you for reaching out to us about your recent purchase of our product! We apologize that you've encountered an issue with your order (order ID #<ORDER_ID>) and we're here to help resolve the problem as quickly and efficiently as possible.

Could you please provide more details regarding the issue you're experiencing? For

✓ Smoke test complete
  Confirm that EMPATHETIC response uses warm/personal language
  and FORMAL response uses procedural/institutional language.


In [17]:
# ─── Cell 12 — Final verification: list saved .pkl files ─────────────────────
#
# CHANGED: note updated — 13b vectors are ~20 KB (5120 float32 values),
#          not ~16 KB (4096 float32 values) as in the 7b notebook.

import os
import pickle
import torch
from pathlib import Path

cache_dir = Path(".style_cache")
pkl_files = sorted(cache_dir.glob("*.pkl"))

print("=" * 60)
print("Stored vectors in .style_cache/")
print("=" * 60)

if not pkl_files:
    print("  (no .pkl files found — did Cells 8 and 9 run successfully?)")
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        with open(pkl_path, "rb") as f:
            v = pickle.load(f)
        print(f"  {pkl_path.name:<45} {size_kb:6.2f} KB   "
              f"shape={list(v.shape)}  norm={v.norm():.4f}")

print("=" * 60)
print(f"Cache directory (absolute): {cache_dir.resolve()}")
print()
print("  Note: ~20 KB each = 5120 float32 values (Llama-2-13b hidden dim)")
print()
print("✓ Vector build pipeline complete.")
print()
print("Next steps:")
print("  • Download the .style_cache/ folder")
print("  • Place it next to main.py in your production environment")
print("  • Set STYLE_CACHE_DIR=.style_cache in your .env file")
print("  • Set LOCAL_MODEL_NAME=meta-llama/Llama-2-13b-chat-hf in your .env")
print("  • Run single scenario:  python main.py battery_issue")
print("  • Run full 100-case batch:  python main.py batch")

Stored vectors in .style_cache/
  style_vec_empathetic_pca.pkl                   20.39 KB   shape=[5120]  norm=1.0000
  style_vec_formal_pca.pkl                       20.39 KB   shape=[5120]  norm=1.0000
Cache directory (absolute): /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-2-13b-chat-hf/.style_cache

  Note: ~20 KB each = 5120 float32 values (Llama-2-13b hidden dim)

✓ Vector build pipeline complete.

Next steps:
  • Download the .style_cache/ folder
  • Place it next to main.py in your production environment
  • Set STYLE_CACHE_DIR=.style_cache in your .env file
  • Set LOCAL_MODEL_NAME=meta-llama/Llama-2-13b-chat-hf in your .env
  • Run single scenario:  python main.py battery_issue
  • Run full 100-case batch:  python main.py batch


In [18]:
# ─── Cell 13 — [Optional] Run the full A2A pipeline (single scenario) ────────
#
# CHANGE vs previous main.py:
#   run_pipeline() now requires three arguments:
#       run_pipeline(scenario_key: str, llm, llm_customer)
#   where `llm` steers the support agent and `llm_customer` generates the complaint.
#
# This cell also explicitly displays the Knowledge Graph panel and the
# Style Comparison table — run_pipeline() no longer calls these internally.
#
# Requires GROQ_API_KEY to be set.
# Scenarios: battery_issue | wrong_item | billing_error | delivery_delay

import asyncio

SCENARIO = "battery_issue"   # ← change to try other scenarios

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠  Skipping: GROQ_API_KEY not set.")
    print("   Set it in Cell 2 or via: export GROQ_API_KEY='gsk_...'")
else:
    print(f"Running full pipeline — scenario: {SCENARIO}")
    print("(Customer complaint is LLM-generated, so output varies each run)")
    print()

    # ── Instantiate LLMs ─────────────────────────────────────────────────────
    # llm          : used by the support agent (deterministic, temp=0.3)
    # llm_customer : used to generate a realistic customer complaint (temp=0.7)
    from langchain_groq import ChatGroq
    llm          = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    llm_customer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

    # Colab already has a running event loop; use nest_asyncio to allow asyncio.run()
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio
        nest_asyncio.apply()

    # ── Run the pipeline ─────────────────────────────────────────────────────
    # run_pipeline now returns a structured record dict in addition to
    # printing the customer panel and both support response panels via Rich.
    record = asyncio.run(
        main_module.run_pipeline(SCENARIO, llm, llm_customer)
    )

    # ── Display Knowledge Graph panel ────────────────────────────────────────
    # run_pipeline() does NOT call make_kg_panel() internally; display it here.
    emp_out = {
        "kg_summary":           record["knowledge_graph"],
        "defactualized_prompt": record["empathetic_output"]["defactualized_prompt"],
    }
    main_module.console.print(main_module.make_kg_panel(emp_out))

    # ── Display style comparison table ───────────────────────────────────────
    # Reconstruct the dicts expected by make_comparison_table.
    emp_response  = {
        "target_style":         record["empathetic_output"]["target_style"],
        "steering_mode":        record["empathetic_output"]["steering_mode"],
        "kg_summary":           record["knowledge_graph"],
        "support_response":     record["empathetic_output"]["support_response"],
    }
    form_response = {
        "target_style":         record["formal_output"]["target_style"],
        "steering_mode":        record["formal_output"]["steering_mode"],
        "kg_summary":           record["knowledge_graph"],
        "support_response":     record["formal_output"]["support_response"],
    }
    main_module.console.print(main_module.make_comparison_table(emp_response, form_response))

    # ── Summary line ─────────────────────────────────────────────────────────
    main_module.console.print(main_module.Rule("[bold]Pipeline complete[/bold]"))
    main_module.console.print(
        f"  Empathetic: [{('green' if emp_response['steering_mode'] == 'kg_activation' else 'yellow')}]"
        f"{emp_response['steering_mode']}[/]\n"
        f"  Formal:     [{('blue'  if form_response['steering_mode'] == 'kg_activation' else 'yellow')}]"
        f"{form_response['steering_mode']}[/]\n"
    )
    main_module.console.print(
        f"[dim]KG nodes: [bold cyan]{len(record['knowledge_graph'].get('nodes', []))}[/bold cyan]  "
        "| Defactualization masked entities before steering  "
        "| Rehydration restored them after generation[/dim]"
    )
    print()
    print(f"Full record keys: {list(record.keys())}")


Running full pipeline — scenario: battery_issue
(Customer complaint is LLM-generated, so output varies each run)



/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out to support for the second time regarding my TechPro X200 Laptop, order number ORD-7741, which │
│ I purchased just three weeks ago. Despite updating the drivers as suggested last week, the battery life issue   │
│ persists - it's still draining completely within 2 hours of a full charge, which is severely impacting my work. │
│ I have a critical deadline to meet tomorrow and urgently need a solution to this problem, so I'd appreciate it  │
│ if we could resolve this today. Could you please assist me with a more permanent fix or let me know the next    │
│ steps to get my laptop functioning properly?"                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ As someone who relies on their technology to work seamlessly,      │
│ having issues like these can cause significant disruptions in your │
│ daily routine. Let me see what I can do to help you find a         │
│ resolution that works best for you. Please hold while I look into  │
│ this further.                                                      │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ As someone who relies on their technology to work seamlessly,      │
│ having issues like these can cause significant disruptions in your │
│ daily routine. Let me see what I can do to help you find a         │
│ resolution t                                                       │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team is actively working on finding a long-term resolution for │
│ the battery life issue you're experiencing with your recent        │
│ purchase of our product (order #). Please be assured that we       │
│ prioritize addressing issues like yours promptly and will do       │
│ everything possible within reason to ensure an optimal user        │
│ experience moving forward. One of our technical specialists will   │
│ reach out shortly via email or phone call to discuss potential     │
│ solutions further and provide any necessary assistance toward      │
│ resolving this matter expeditiously.                               │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team is actively working on finding a long-term resolution for │
│ the battery life issue you're experiencing with your recent        │
│ purchase of our product (order #). Please be assured that we       │
│ prioritize a                                                       │
╰────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Knowledge Graph ────────────────────────────────────────────────╮
│ Nodes (6):                                                                                                      │
│   (PRODUCT) TechPro X200 Laptop  (salience: 0.8)                                                                │
│   (ORDER_ID) ORD-7741  (salience: 0.9)                                                                          │
│   (ISSUE) persists - it's still draining completely within 2 hours of a full charge, which is severely          │
│ impacting my work  (salience: 0.9)                                                                              │
│   (URGENCY) high  (salience: 1.0)                                                                               │
│   (SENTIMENT) polite  (salience: 0.7)                                                                           │
│   (CUSTOMER_NAME) Priya Sharma  (salience: 0.7)                                                                 │
│                                                                                                                 │
│ Edges (4):                                                                                                      │
│   TechPro X200 Laptop       —has_issue→ persists - it's still dra                                               │
│   ORD-7741                  —about_product→ TechPro X200 Laptop                                                 │
│   Priya Sharma              —placed_order→ ORD-7741                                                             │
│   persists - it's still dra —has_urgency→ high                                                                  │
│                                                                                                                 │
│ Defactualized prompt:                                                                                           │
│   I'm reaching out to support for the second time regarding my <PRODUCT>, order number <ORDER_ID>, which I      │
│ purchased just three weeks ago. Despite updating the drivers as suggested last week, the batter                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Style Comparison — Same Facts, Different Tone                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension       ┃ Empathetic Pipeline                           ┃ Formal Pipeline                               ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Target style    │ empathetic                                    │ formal                                        │
│ Steering mode   │ kg_activation                                 │ kg_activation                                 │
│ KG nodes        │ 6                                             │ 6                                             │
│ KG edges        │ 4                                             │ 4                                             │
│ Response (first │ As someone who relies on their technology to  │ Our team is actively working on finding a     │
│ 200 chars)      │ work seamlessly, having issues like these can │ long-term resolution for the battery life     │
│                 │ cause significant disruptions in your daily   │ issue you're experiencing with your recent    │
│                 │ routine. Let me see what I can do to help you │ purchase of our product (order #). Please be  │
│                 │ find a resolution t                           │ assured that we prioritize a                  │
└─────────────────┴───────────────────────────────────────────────┴───────────────────────────────────────────────┘

──────────────────────────────────────────────── Pipeline complete ────────────────────────────────────────────────

Empathetic: kg_activation
  Formal:     kg_activation

KG nodes: 6  | Defactualization masked entities before steering  | Rehydration restored them after generation


Full record keys: ['input', 'knowledge_graph', 'empathetic_output', 'formal_output']


In [19]:
# ─── Cell 14 — [Optional] Run the full 100-case batch ─────────────────────────
#
# NEW in updated main.py: run_batch() iterates all 100 BATCH_CASES, calls
# run_pipeline() for each, and writes results to:
#   outputs/results_<timestamp>.jsonl   — one JSON record per case
#   outputs/knowledge_graphs/           — one KG JSON file per case
#
# run_batch() creates its own ChatGroq LLM instances internally, so you only
# need GROQ_API_KEY to be set.
#
# ⚠  This takes ~30–90 min depending on Groq rate limits and activation-steering
#    availability. Each case runs sequentially (not in parallel) to avoid
#    overwhelming the API.
#
# To run a subset, slice BATCH_CASES before calling run_batch():
#   main_module.BATCH_CASES[:10]   # first 10 only

import asyncio

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠  Skipping: GROQ_API_KEY not set.")
    print("   Set it in Cell 2 or via: export GROQ_API_KEY='gsk_...'")
else:
    # Verify output directories will be created
    from pathlib import Path
    Path("outputs_spacy").mkdir(exist_ok=True)
    Path("outputs_spacy/knowledge_graphs").mkdir(exist_ok=True)

    print(f"Starting batch run — {len(main_module.BATCH_CASES)} cases")
    print(f"Results will be written to: outputs_spacy/results_<timestamp>.jsonl")
    print(f"KG files will be written to: outputs_spacy/knowledge_graphs/")
    print()

    # Ensure nest_asyncio is applied for Colab/Jupyter environments
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio
        nest_asyncio.apply()

    # ── Optional: run a subset only ───────────────────────────────────────────
    # Uncomment to limit the run (useful for testing):
    # _original_batch = main_module.BATCH_CASES
    # main_module.BATCH_CASES = main_module.BATCH_CASES[:5]  # first 5 cases only

    asyncio.run(main_module.run_batch())

    # Restore if you sliced above:
    # main_module.BATCH_CASES = _original_batch


Starting batch run — 100 cases
Results will be written to: outputs_spacy/results_<timestamp>.jsonl
KG files will be written to: outputs_spacy/knowledge_graphs/



──────────────────────────────────── A2A Customer Support — 100-Case Batch Run ────────────────────────────────────

Output file : outputs/results_20260524_024145.jsonl
  Total cases : 100
  Activation steering: ENABLED

──────────────── Case   1/100  scenario=battery_issue  customer=Priya Sharma  sentiment=frustrated ────────────────

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out again regarding my TechPro X200 Laptop, order number ORD-7741, which I purchased just three   │
│ weeks ago. As I mentioned last week, the battery is draining completely within 2 hours of a full charge, which  │
│ is really impacting my freelance design work. I followed the advice from my previous support call and updated   │
│ the drivers, but unfortunately, the issue persists. With a critical client deadline looming tomorrow, I         │
│ urgently need a solution to this problem today, so I'd appreciate it if we could explore other options to       │
│ resolve this as soon as possible."                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Ximena Flores   Order: ORD-8858                                                                       │
│ Product: Ergonomic Office Chair                                                                                 │
│ Issue: delivery now 3 weeks overdue with no update from courier                                                 │
│ Sentiment: NEUTRAL   Urgency: medium                                                                            │
│                                                                                                                 │
│ "I'm reaching out about my order, ORD-8858, for the Ergonomic Office Chair, which was supposed to arrive three  │
│ weeks ago. Unfortunately, I still haven't received it and haven't gotten any updates from the courier. I've     │
│ been working on the floor for weeks now, so I'd appreciate it if you could provide a clear timeline for when I  │
│ can expect the chair to arrive or offer a refund so I can look into purchasing one locally. Could you please    │
│ look into this and let me know my options?"                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ We value your business and want to make sure that we resolve this  │
│ issue as soon as possible. Our team is actively looking into what  │
│ might have caused the delay in shipping and will keep you updated  │
│ with regular progress reports until your order arrives safely at   │
│ your doorstep. Please bear with us just a little longer while we   │
│ work through this together.                                        │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ We value your business and want to make sure that we resolve this  │
│ issue as soon as possible. Our team is actively looking into what  │
│ might have caused the delay in shipping and will keep you updated  │
│ wi                                                                 │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team is actively investigating the matter with our shipping    │
│ partner to determine why there have been no updates regarding      │
│ delivery status since purchase; we will keep you informed via      │
│ email once an update becomes available as soon as possible         │
│ concerning estimated arrival dates or potential compensation       │
│ solutions such as refunds or replacements should they become       │
│ necessary due to delays beyond control by either party involved in │
│ fulfilling orders successfully within agreed timeframes set forth  │
│ between ourselves & customers like yourself who rely upon us       │
│ providing quality products promptly without fail every single      │
│ time!                                                              │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team is actively investigating the matter with our shipping    │
│ partner to determine why there have been no updates regarding      │
│ delivery status since purchase; we will keep you informed via      │
│ email once a                                                       │
╰────────────────────────────────────────────────────────────────────╯

✓ Saved case 100  |  KG → knowledge_graphs/kg_100_delivery_delay_ORD-8858_Ximena_Flores.json

───────────────────────────────────────────────── Batch complete ──────────────────────────────────────────────────

Success: 100/100
  Failed:  0/100
  Results: outputs/results_20260524_024145.jsonl
  KGs:     outputs/knowledge_graphs/  (100 files)